# RoBERTa fine-tune

Runs `complaint.train` on a Colab GPU. Before running:
- push the latest code to GitHub (the notebook clones it)
- upload `train/val/test.parquet` to Google Drive, under `DATA_DIR` below

Set `RUN = "smoke"` first to check the pipeline end to end, then `"full"`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

In [ ]:
RUN = "smoke"   # "smoke" or "full"

REPO_URL = "https://github.com/leoamullins/complaint-triage.git"
REPO_DIR = "/content/complaint-triage"
DATA_DIR = "/content/drive/MyDrive/complaint-triage"
OUTPUT_ROOT = "/content/drive/MyDrive/complaint-outputs"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

# torch comes with colab; only add what the project needs on top
%pip install -q "transformers>=5.20.0" "datasets>=5.1.0" "accelerate>=1.1.0" pyyaml pyarrow scikit-learn

In [ ]:
import sys
sys.path.insert(0, f"{REPO_DIR}/src")

from pathlib import Path
from complaint.config import Config, load_config

if RUN == "smoke":
    cfg = load_config(f"{REPO_DIR}/configs/smoke.yaml")
else:
    cfg = Config()   # roberta-base, max_len 512, 2 epochs, bf16

cfg.data.data_dir = DATA_DIR
cfg.output_dir = f"{OUTPUT_ROOT}/{cfg.run_name}"

assert all(Path(DATA_DIR, f"{s}.parquet").exists() for s in ("train", "val", "test")), f"parquet files missing from {DATA_DIR}"
cfg

In [ ]:
from complaint.train import train

trainer, dfs, ds, log_ratio = train(cfg)

## Validation results

In [ ]:
from sklearn.metrics import classification_report, f1_score
from complaint.evaluation import adjusted_preds

id2label = trainer.model.config.id2label
names = [id2label[i] for i in range(len(id2label))]

val_out = trainer.predict(ds["val"])
y = val_out.label_ids
raw = val_out.predictions.argmax(-1)
adj = adjusted_preds(val_out.predictions, log_ratio, cfg.evaluation.tau)

print("macro-F1 raw:     ", f1_score(y, raw, average="macro"))
print(f"macro-F1 tau={cfg.evaluation.tau}:", f1_score(y, adj, average="macro"))
print(classification_report(y, adj, target_names=names, digits=3))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y, adj, display_labels=names, xticks_rotation=90, normalize="true"
)